# 4.7 · 广义线性模型 / Generalized Linear Models (GLM)

> **课程定位 / Where this fits**
> **Part 4 第 7 课**。前 6 课目标 y 都默认**高斯**（连续、对称、值域无限）。但现实里 y 常是**计数**（理赔次数→泊松）、**二值**（是否点击→伯努利）、**正偏态金额**（理赔额→伽马）。GLM 用 **link function** 把线性模型优雅推广到整个**指数族**。它是逻辑回归(Part 5)、泊松回归的统一框架。
> Real targets are often counts (Poisson), binary (Bernoulli), or skewed amounts (Gamma). GLM generalizes the linear model to the exponential family via a link function — the unifying frame for logistic and Poisson regression.

> 💡 **面试相关 / Interview-relevant**
> - "什么时候不能用普通线性回归" ★★★★（y 非高斯）
> - "GLM 的三要素" ★★★★（随机分量+线性预测器+link）
> - "为什么计数数据用泊松回归不用 OLS" ★★★★
> - "逻辑回归是 GLM 吗" ★★★★（是, logit link 的伯努利 GLM）

---

## 学习目标 / Learning Objectives
1. 理解普通线性回归对**非高斯 y** 的失败（预测负计数、异方差）。
2. 掌握 GLM **三要素**：随机分量(分布) + 线性预测器 + link function。
3. 用**泊松回归**建模计数, **伽马回归**建模正偏态金额。
4. 理解 GLM 把线性回归 / 逻辑回归 / 泊松回归统一起来。

## 目录 / TOC
1. [线性回归对非高斯 y 的失败 ⭐](#1)
2. [GLM 三要素 ⭐](#2)
3. [常见 GLM 一览表](#3)
4. [泊松回归: 建模计数 ⭐](#4)
5. [伽马回归: 建模正偏态金额](#5)
6. [GLM 统一视角](#6)
7. [小结](#7)


<a id="1"></a>
## 1. 线性回归对非高斯 y 的失败 ⭐ / Where OLS Fails

普通线性回归 $\hat{y} = \mathbf{x}^\top\mathbf{w}$ 隐含假设（4.1 节）：y 是**高斯**——连续、对称、值域 $(-\infty, +\infty)$、**同方差**。

对这些 y 它会出错：

| y 的类型 | OLS 的问题 |
|---|---|
| **计数**（0,1,2,...理赔次数）| 预测出**负数**或小数（无意义）; 计数天然异方差(均值=方差, 2.2 泊松) |
| **二值**（0/1 点击）| 预测出 < 0 或 > 1 的"概率"（→ Part 5 逻辑回归）|
| **正偏态金额**（理赔额, 必 >0, 右偏）| 高斯对称假设错; 预测负金额 |

**根本问题**: OLS 把 y 的**期望直接**等于线性组合 $\mu = \mathbf{x}^\top\mathbf{w}$, 但 $\mathbf{x}^\top\mathbf{w}$ 值域无限, 而很多 y 的期望有约束（计数≥0, 概率∈[0,1]）。
OLS sets E[y] directly equal to an unbounded linear combination — but many targets' means are constrained.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
sns.set_theme(style="whitegrid")
rng = np.random.default_rng(42)

# 造计数数据: 理赔次数随风险评分增加 (泊松) / claim counts grow with risk
n = 1000
risk = rng.uniform(0, 3, n)
true_rate = np.exp(-0.5 + 0.8*risk)        # 真实: log(率) 线性于 risk
claims = rng.poisson(true_rate)             # 泊松计数 / Poisson counts

# OLS 直接拟合 / fit OLS directly
ols = sm.OLS(claims, sm.add_constant(risk)).fit()
x_plot = np.linspace(0, 3, 100)
ols_pred = ols.predict(sm.add_constant(x_plot))

fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(risk, claims, alpha=0.2, s=10, label="理赔次数(计数)")
ax.plot(x_plot, ols_pred, "r-", lw=2, label="OLS 拟合")
ax.axhline(0, color="k", lw=0.5)
ax.set_xlabel("risk score"); ax.set_ylabel("claims"); ax.legend()
ax.set_title("OLS 对计数数据: 低 risk 处预测负理赔次数 (无意义!)")
plt.tight_layout(); plt.show()
print(f"OLS 在 risk=0 处预测理赔次数 = {ols_pred[0]:.2f}  ← 负数! 计数不可能为负")
print("且计数数据方差随均值增大 (泊松性质) → OLS 同方差假设也违反")


<a id="2"></a>
## 2. GLM 三要素 ⭐ / The Three Components

GLM 优雅地推广线性模型, 由**三个部件**组成：

| 部件 | 作用 | 例子 |
|---|---|---|
| **1. 随机分量** Random component | y 服从**指数族**某分布 | 高斯/伯努利/泊松/伽马 |
| **2. 线性预测器** Linear predictor | $\eta = \mathbf{x}^\top\mathbf{w}$（值域无限）| 永远是线性 |
| **3. link function** $g$ | 连接均值与线性预测器: $g(\mu) = \eta$ | 把有约束的 μ 映到无限的 η |

**核心机制**: 不再 $\mu = \mathbf{x}^\top\mathbf{w}$, 而是 $g(\mu) = \mathbf{x}^\top\mathbf{w}$, 即 $\mu = g^{-1}(\mathbf{x}^\top\mathbf{w})$。

**link 的妙处**: 它把"有约束的均值"映射到"无约束的线性空间"。例如泊松用 **log link**: $\log\mu = \mathbf{x}^\top\mathbf{w}$ → $\mu = e^{\mathbf{x}^\top\mathbf{w}} > 0$ **永远为正**, 计数预测合理。
The link maps a constrained mean to the unconstrained linear space. Poisson's log link guarantees a positive predicted count.


<a id="3"></a>
## 3. 常见 GLM 一览表 / The GLM Family

| y 类型 | 分布 | 典型 link | $\mu$ 范围 | 模型名 | 出现在 |
|---|---|---|---|---|---|
| 连续对称 | 高斯 | identity $\mu=\eta$ | $\mathbb{R}$ | **普通线性回归** | 4.1 |
| 二值 | 伯努利 | logit $\log\frac{\mu}{1-\mu}$ | $[0,1]$ | **逻辑回归** | Part 5.1 |
| 计数 | 泊松 | log $\log\mu$ | $[0,\infty)$ | **泊松回归** | 本课 |
| 正偏态(>0) | 伽马 | log | $(0,\infty)$ | **伽马回归** | 本课 |
| 计数(过散) | 负二项 | log | $[0,\infty)$ | 负二项回归 | (3.2 提过) |

**统一之美**: 这五个看似不同的模型, **本质是同一个 GLM 框架换不同的(分布, link)组合**。逻辑回归不是独立发明, 是 GLM 的伯努利+logit 特例。
The unifying beauty: all of these are the same GLM with different (distribution, link) pairs. Logistic regression is just the Bernoulli+logit case.


<a id="4"></a>
## 4. 泊松回归: 建模计数 ⭐ / Poisson Regression

用 statsmodels 的 GLM, 指定 `family=Poisson()`（默认 log link）。


In [ ]:
# 泊松 GLM / Poisson GLM
poisson_glm = sm.GLM(claims, sm.add_constant(risk),
                     family=sm.families.Poisson()).fit()
poisson_pred = poisson_glm.predict(sm.add_constant(x_plot))

fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(risk, claims, alpha=0.2, s=10, label="理赔次数")
ax.plot(x_plot, ols.predict(sm.add_constant(x_plot)), "r--", lw=1.5, label="OLS (会预测负数)")
ax.plot(x_plot, poisson_pred, "g-", lw=2.5, label="泊松回归 (永远>0)")
ax.plot(x_plot, np.exp(-0.5+0.8*x_plot), "k:", lw=1.5, label="真实率")
ax.axhline(0, color="k", lw=0.5); ax.legend()
ax.set_xlabel("risk"); ax.set_ylabel("claims")
ax.set_title("泊松回归: 指数曲线, 永远为正, 贴合真实")
plt.tight_layout(); plt.show()

print(f"泊松回归系数: 截距={poisson_glm.params[0]:.3f} (真-0.5), risk={poisson_glm.params[1]:.3f} (真0.8)")
print(f"→ 准确恢复真实参数!")
print(f"\n💡 系数解读 (log link): risk 每+1, 理赔率 ×e^{poisson_glm.params[1]:.2f}={np.exp(poisson_glm.params[1]):.2f} 倍")
print("   (log link 下系数是'乘性'效应, 不是加性 — 这是泊松回归的标准解读)")


<a id="5"></a>
## 5. 伽马回归: 建模正偏态金额 / Gamma Regression

理赔**金额**（不是次数）: 必 > 0, 右偏（多数小额, 少数巨额）, 方差随均值增大。**伽马分布 + log link** 是标准选择（保险精算核心）。


In [ ]:
# 造理赔金额数据: 正偏态, 随 risk 增大 / claim amounts, right-skewed
true_mean_amount = np.exp(6 + 0.5*risk)               # log(均值) 线性
amount = rng.gamma(shape=2.0, scale=true_mean_amount/2.0)  # 伽马, 均值=shape*scale

print(f"理赔金额: 全部>0, 右偏 (skew={pd.Series(amount).skew():.1f})")

# 伽马 GLM vs OLS / Gamma GLM vs OLS
gamma_glm = sm.GLM(amount, sm.add_constant(risk),
                   family=sm.families.Gamma(link=sm.families.links.Log())).fit()
ols_amt = sm.OLS(amount, sm.add_constant(risk)).fit()

gpred = gamma_glm.predict(sm.add_constant(x_plot))
opred = ols_amt.predict(sm.add_constant(x_plot))

fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(risk, amount, alpha=0.15, s=10, label="理赔金额")
ax.plot(x_plot, opred, "r--", lw=1.5, label="OLS")
ax.plot(x_plot, gpred, "g-", lw=2.5, label="伽马回归")
ax.plot(x_plot, np.exp(6 + 0.5*x_plot), "k:", lw=1.5, label="真实均值")
ax.legend(); ax.set_xlabel("risk"); ax.set_ylabel("claim amount")
ax.set_title("伽马回归: 捕捉指数增长 + 正偏态")
plt.tight_layout(); plt.show()
print("伽马回归对右偏正值金额更合适; OLS 被极端大额拉偏 + 可能预测负金额")
print("→ 保险定价标准: 理赔频率用泊松, 理赔金额用伽马 (合起来是 Tweedie 模型)")


<a id="6"></a>
## 6. GLM 统一视角 / The Unifying View

验证: **GLM 用 identity link + 高斯 = 普通线性回归**, 说明 4.1 只是 GLM 的一个特例。


In [ ]:
from sklearn.datasets import fetch_california_housing
from sklearn.linear_model import LinearRegression

d = fetch_california_housing(as_frame=True)
Xh, yh = d.data.values[:2000], d.target.values[:2000]

# GLM(高斯+identity) vs sklearn LinearRegression / should be identical
glm_gaussian = sm.GLM(yh, sm.add_constant(Xh), family=sm.families.Gaussian()).fit()
lr = LinearRegression().fit(Xh, yh)

print(f"GLM(高斯,identity) 系数 vs LinearRegression 系数:")
print(f"最大差异 = {np.abs(glm_gaussian.params[1:] - lr.coef_).max():.8f}")
print("→ 完全相同! 普通线性回归 = GLM 的(高斯, identity link)特例")
print("\nGLM 把一切统一: 改(分布,link)就得到不同模型, 同一套 IRLS 算法拟合")


<a id="7"></a>
## 7. 小结 / Summary

```
OLS 失败场景: y 非高斯 (计数→负数, 二值→越界概率, 金额→负值/偏态)
GLM 三要素 ⭐:
  随机分量 (指数族分布) + 线性预测器 η=xᵀw + link g(μ)=η
  核心: μ = g⁻¹(xᵀw), link 把有约束的均值映到无约束线性空间
GLM 家族:
  高斯+identity = 线性回归 | 伯努利+logit = 逻辑回归(Part5)
  泊松+log = 计数 | 伽马+log = 正偏态金额
log link 系数 = 乘性效应 (x+1 → 率×e^β)
保险定价: 频率泊松 + 金额伽马 (= Tweedie)
```

### 💡 面试速查
1. **y 非高斯就别用 OLS**: 计数/二值/偏态金额
2. **GLM 三要素**: 分布 + 线性预测器 + link
3. **逻辑回归是 GLM** (伯努利+logit); 泊松回归(计数)
4. **log link 系数是乘性的** (e^β 倍)
5. **OLS = GLM(高斯+identity)特例** — 一切统一

### 下一节
**4.8 非线性回归**——GLM 仍是"广义线性"(对参数线性)。真正参数非线性的模型（如 $y=a e^{bx}$ 中 a,b 缠绕）需要 `scipy.optimize` 的 Levenberg-Marquardt。
